# ETL e Preparação de Dados

**Objetivos:** construir um fluxo reprodutível de extração, validação, transformação, integração e exportação; compreender escolhas de tratamento e reconhecer riscos de perda de informação e vazamento de dados.

Os exemplos usam dados **sintéticos** em `datasets/` e gravam somente resultados derivados em `outputs/`. Execute as células em ordem a partir da pasta da biblioteca. As duas pastas são criadas automaticamente, mas os arquivos CSV precisam existir em `datasets/`.

> **Princípio:** não altere a fonte original. Toda exclusão, imputação ou transformação deve ser motivada, verificável e, quando possível, reversível.

### Materiais complementares

- [Guia Técnico](./reference_materials/02_etl_e_preparacao_de_dados.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_etl.pdf)

Os arquivos estão nesta mesma pasta desta área. Os PDFs complementam, mas não substituem, a execução das células.

## 1. Visão geral de ETL

- **Extrair:** carregar fontes respeitando esquema, codificação e caminho.
- **Transformar:** checar tipos, nulos, duplicatas e regras de negócio; integrar tabelas.
- **Carregar:** disponibilizar os resultados e confirmar que podem ser lidos novamente.

Neste exercício, arquivos de entrada permanecem em `datasets/`; os arquivos produzidos ficam em `outputs/`. O fluxo usa `vendas_exemplo.csv` e `regioes_exemplo.csv`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def preparar_pastas():
    """Localiza a biblioteca e cria as pastas de dados e resultados, se ausentes."""
    atual = Path.cwd().resolve()
    # Aceita execução na raiz da biblioteca ou em uma pasta filha.
    candidatos = [atual, atual.parent]
    raiz = next(
        (p for p in candidatos if (p / 'vendas_exemplo.csv').is_file() or
         (p / 'datasets' / 'vendas_exemplo.csv').is_file()),
        None,
    )
    if raiz is None:
        # Identifica a raiz da biblioteca pelo conjunto de notebooks.
        raiz = next(
            (p for p in candidatos if (p / '01_Analise_Exploratoria_de_Dados.ipynb').is_file()),
            atual,
        )
    dados = raiz / 'datasets'
    saidas = raiz / 'outputs'
    dados.mkdir(parents=True, exist_ok=True)
    saidas.mkdir(parents=True, exist_ok=True)
    return dados, saidas

DATA, OUTPUTS = preparar_pastas()

def conferir_datasets(*nomes):
    ausentes = [nome for nome in nomes if not (DATA / nome).is_file()]
    if ausentes:
        raise FileNotFoundError(
            'Datasets ausentes: ' + ', '.join(ausentes) + '\n'
            + 'Coloque os CSVs fornecidos em: ' + str(DATA.resolve()) + '\n'
            + 'Criar a pasta datasets/ não cria automaticamente os arquivos CSV.'
        )
conferir_datasets('vendas_exemplo.csv', 'regioes_exemplo.csv')
brutos=pd.read_csv(DATA/'vendas_exemplo.csv',parse_dates=['data'])
print(brutos.shape)
print(brutos.isna().sum())

(49, 6)
data              0
regiao            1
produto           0
unidades          0
preco_unitario    1
canal             0
dtype: int64


## 2. Transformações e validações explícitas

Antes de transformar, verifique se cada linha representa uma venda e se registros aparentemente duplicados são de fato repetições do mesmo evento. **Neste conjunto didático**, removemos duplicatas exatas como hipótese de tratamento documentada; em um sistema real precisaríamos de um identificador de transação.

Região ausente será rotulada `Não informado`. Preço ausente permanece **desconhecido (`NaN`)**: isso não significa preço ou receita zero.

In [2]:
duplicatas_exatas = int(brutos.duplicated().sum())
tratados = brutos.drop_duplicates().copy()
tratados['regiao'] = tratados['regiao'].fillna('Não informado')
tratados['receita'] = tratados['unidades'] * tratados['preco_unitario']
assert tratados['unidades'].notna().all() and tratados['unidades'].ge(0).all()
assert tratados['data'].notna().all()
assert len(brutos) - len(tratados) == duplicatas_exatas
print('Linhas de entrada:', len(brutos), '| duplicatas exatas:', duplicatas_exatas, '| tratadas:', len(tratados))
print('Registros sem preço/receita calculável:', tratados['receita'].isna().sum())

Linhas de entrada: 49 | duplicatas exatas: 1 | tratadas: 48
Registros sem preço/receita calculável: 1


## 3. Integração de tabelas e chaves

Uma junção `many_to_one` exige que cada região apareça no máximo uma vez na tabela de referência. O parâmetro `indicator=True` registra a origem da correspondência (`both`, `left_only` ou `right_only`). A categoria `Não informado` não tem referência — isso é um alerta, não um motivo automático para remover a venda.

In [3]:
referencia=pd.read_csv(DATA/'regioes_exemplo.csv')
integrado=tratados.merge(referencia,on='regiao',how='left',validate='many_to_one',indicator=True)
print(integrado['_merge'].value_counts().to_string())
print(integrado[['regiao','responsavel','meta_mensal']].head())

_merge
both          47
left_only      1
right_only     0
     regiao responsavel  meta_mensal
0     Norte    Equipe A        600.0
1       Sul    Equipe B        700.0
2   Sudeste    Equipe C        850.0
3  Nordeste    Equipe D        500.0
4       Sul    Equipe B        700.0


## 4. Exercício resolvido — resumir receita por mês e produto

**Enunciado:** gere uma tabela mensal com a receita **conhecida** por produto, quantidade de registros e quantidade de receitas ausentes. Preserve o diagnóstico de incompletude para que o total não seja confundido com a receita real completa. A exportação será feita na seção 6.

In [4]:
saida = integrado.copy()
saida['mes'] = saida['data'].dt.to_period('M').astype(str)
resumo = (saida.groupby(['mes', 'produto'], as_index=False)
          .agg(receita_conhecida=('receita', lambda x: x.sum(min_count=1)),
               registros=('receita', 'size'),
               receitas_ausentes=('receita', lambda x: int(x.isna().sum()))))
assert int(resumo['registros'].sum()) == len(saida)
assert int(resumo['receitas_ausentes'].sum()) == int(saida['receita'].isna().sum())
print(resumo.head())
print('Receita conhecida não significa receita total quando existem preços ausentes.')

       mes produto  receita_conhecida  registros  receitas_ausentes
0  2024-01    Café              342.0          1                  0
1  2024-01     Chá              532.0          1                  0
2  2024-01    Suco                NaN          1                  1
3  2024-01    Água              357.0          1                  0
4  2024-02    Café              558.0          1                  0
Receita conhecida não significa receita total quando existem preços ausentes.


## 5. ETL reprodutível: esquema, validações e falhas controladas

Siga a ordem: extrair → conferir tipos/chaves → transformar **sem destruir a fonte** → validar → carregar. Demonstre falhas controladas com mensagens legíveis. `assert` é útil em exercícios, mas aplicações devem usar exceções explícitas para validar dados de entrada.

In [5]:
def validar_vendas(tabela):
    obrigatorias = {'data', 'regiao', 'produto', 'unidades', 'preco_unitario'}
    faltando = obrigatorias - set(tabela.columns)
    if faltando:
        raise ValueError(f'Colunas ausentes: {sorted(faltando)}')
    if tabela['unidades'].isna().any() or tabela['unidades'].lt(0).any():
        raise ValueError('Unidades ausentes ou negativas não são permitidas')
    if tabela['data'].isna().any():
        raise ValueError('Datas inválidas ou ausentes')
    if tabela['produto'].isna().any():
        raise ValueError('Produto é obrigatório')
    return True

assert validar_vendas(brutos)
try:
    validar_vendas(pd.DataFrame({'produto': ['Chá']}))
except ValueError as exc:
    print('Falha controlada:', exc)

Falha controlada: Colunas ausentes: ['data', 'preco_unitario', 'regiao', 'unidades']


## 6. Carregamento em `outputs/` e leitura de retorno

Um ETL deve verificar se o arquivo exportado pode ser recuperado e se os totais relevantes foram preservados. Esta célula cria/sobrescreve somente `outputs/receita_mensal.csv`; não altera os CSVs de origem.

In [6]:
destino = OUTPUTS / 'receita_mensal.csv'
resumo.to_csv(destino, index=False, encoding='utf-8')
retorno = pd.read_csv(destino)
assert retorno.shape == resumo.shape
assert retorno['registros'].sum() == resumo['registros'].sum()
assert retorno['receitas_ausentes'].sum() == resumo['receitas_ausentes'].sum()
print(f'Exportação verificada: {destino.resolve()} ({len(retorno)} linhas)')

Exportação verificada: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/02_data_analysis_visualization/outputs/receita_mensal.csv (48 linhas)


## 7. Exercício resolvido — auditoria de integridade

**Enunciado:** verifique unicidade das chaves na referência e a faixa esperada de unidades vendidas. Justifique os limites adotados e mostre o número de registros que violariam a regra antes de qualquer exclusão.

In [7]:
assert referencia['regiao'].is_unique, 'Referência com regiões duplicadas'
fora_da_faixa = ~tratados['unidades'].between(0, 10_000)
print('Registros fora da faixa esperada:', int(fora_da_faixa.sum()))
assert not fora_da_faixa.any()
print('Regras verificadas: chaves únicas na referência e unidades na faixa esperada.')

Registros fora da faixa esperada: 0
Regras verificadas: chaves únicas na referência e unidades na faixa esperada.


## 8. Exercício resolvido — integrar, auditar e exportar

**Enunciado:** crie um relatório por região incluindo contagem de registros, receita conhecida e registros sem receita calculável; confira a tabela de referência e identifique regiões sem correspondência. Grave apenas o relatório derivado em `outputs/`.

In [8]:
assert referencia['regiao'].is_unique, 'Referência com chaves duplicadas'
relatorio_regioes = (integrado.groupby('regiao', dropna=False, as_index=False)
    .agg(registros=('produto', 'size'),
         receita_conhecida=('receita', lambda x: x.sum(min_count=1)),
         receitas_ausentes=('receita', lambda x: int(x.isna().sum()))))
sem_referencia = integrado.loc[integrado['_merge'] != 'both', 'regiao'].drop_duplicates().tolist()
assert int(relatorio_regioes['registros'].sum()) == len(integrado)
print('Regiões sem correspondência:', sem_referencia)
print(relatorio_regioes.to_string(index=False))
relatorio_regioes.to_csv(OUTPUTS / 'resumo_por_regiao.csv', index=False)
assert (OUTPUTS / 'resumo_por_regiao.csv').exists()

Regiões sem correspondência: ['Não informado']
       regiao  registros  receita_conhecida  receitas_ausentes
     Nordeste         11             6243.0                  0
        Norte         12             6976.0                  0
Não informado          1              900.0                  0
      Sudeste         12             6589.0                  1
          Sul         12             6986.0                  0


## 9. Tratamentos complementares e engenharia de atributos

Os exemplos seguintes demonstram imputação, discretização, datas, detecção de anomalias e transformações para modelagem. Essa parte é complementar ao ETL de vendas. **Transformações aprendidas a partir dos dados devem ser ajustadas somente no treino** para evitar vazamento de informação do conjunto de teste.

### 9.1. Dados de demonstração e valores ausentes

Escolha estratégias de tratamento considerando mecanismo de ausência, significado das variáveis e objetivo do modelo. `dropna()` elimina registros; `fillna()` substitui valores; `ffill()` propaga a última observação válida — abordagem que só faz sentido quando existe uma ordem relevante.

In [9]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    'pessoa': ['Ana', 'Bia', 'Caio', 'Ana', 'Davi', 'Eva'],
    'idade': [25, np.nan, 33, 25, 42, np.nan],
    'renda': [3000, 4500, 5200, 3000, 8000, 4000],
    'grupo': ['A', 'B', 'B', 'A', 'C', 'A'],
    'data': ['2024-01-05', '2024-02-10', '2024-03-11', '2024-01-05', '2024-04-09', '2024-05-14']
})
print(df.isna().sum())
print(df.head())

pessoa    0
idade     2
renda     0
grupo     0
data      0
dtype: int64
  pessoa  idade  renda grupo        data
0    Ana   25.0   3000     A  2024-01-05
1    Bia    NaN   4500     B  2024-02-10
2   Caio   33.0   5200     B  2024-03-11
3    Ana   25.0   3000     A  2024-01-05
4   Davi   42.0   8000     C  2024-04-09


In [10]:
com_media = df.copy()
com_media['idade'] = com_media['idade'].fillna(com_media['idade'].mean())
assert com_media['idade'].isna().sum() == 0

sem_ausentes = df.dropna(subset=['idade'])
assert len(sem_ausentes) == 4

# Exemplos distintos: propagar o último valor válido somente se existir ordem coerente.
serie_temporal = pd.Series([10.0, np.nan, 12.0, np.nan]).ffill()
assert serie_temporal.tolist() == [10.0, 10.0, 12.0, 12.0]

### 9.2. Duplicações

`duplicated()` identifica duplicações e `drop_duplicates()` as remove. Antes de eliminar registros, defina quais colunas representam a identidade de um evento.

In [11]:
print('Registros completamente duplicados:', int(df.duplicated().sum()))
unicos = df.drop_duplicates()
assert len(unicos) == 5
print('Registros únicos:', len(unicos))

Registros completamente duplicados: 1
Registros únicos: 5


### 9.3. Discretização e categorias

`pd.cut()` cria faixas com limites definidos; `pd.qcut()` tenta dividir observações por quantis. As faixas devem ter interpretação útil. Para usar faixas calculadas a partir dos dados em modelagem, aprenda a ajustá-las sem vazamento entre treino e teste.

In [12]:
idades = pd.Series([18, 23, 31, 45, 62])
faixas = pd.cut(idades, bins=[0, 25, 40, 120], labels=['jovem', 'adulto', 'sênior'])
assert faixas.tolist() == ['jovem', 'jovem', 'adulto', 'sênior', 'sênior']
print(faixas.to_list())

['jovem', 'jovem', 'adulto', 'sênior', 'sênior']


### 9.4. Datas e atributos temporais

`pd.to_datetime()` converte strings para datas. Datas e fusos exigem atenção: `tz_localize()` atribui um fuso a timestamps ingênuos; `tz_convert()` converte instantes já localizados.

In [13]:
com_datas = df.copy()
com_datas['data'] = pd.to_datetime(com_datas['data'])
com_datas['mes'] = com_datas['data'].dt.month
com_datas['dia_semana'] = com_datas['data'].dt.dayofweek
assert com_datas['mes'].iloc[0] == 1

instante_utc = pd.Timestamp('2024-01-01 12:00:00', tz='UTC')
instante_sp = instante_utc.tz_convert('America/Sao_Paulo')
print(instante_utc, instante_sp)

2024-01-01 12:00:00+00:00 2024-01-01 09:00:00-03:00


/tmp/ipykernel_23670/1123156643.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  com_datas['dia_semana'] = com_datas['data'].dt.dayofweek


### 9.5. Detecção de outliers

Valores extremos podem ser erros ou observações legítimas. Métodos como IQR e z-score são referências, não decisões automáticas de remoção. Detectores como DBSCAN e Isolation Forest exigem cuidado com escala, parâmetros e interpretação.

In [14]:
serie = pd.Series([10, 11, 11, 12, 12, 13, 14, 60])
q1, q3 = serie.quantile([0.25, 0.75])
iqr = q3 - q1
limite_inferior, limite_superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = serie[(serie < limite_inferior) | (serie > limite_superior)]
assert 60 in outliers.values
print('Outliers pelo IQR:', outliers.tolist())

Outliers pelo IQR: [60]


### 9.6. Scikit-learn: imputação, escala e anomalias

`SimpleImputer` calcula valores de substituição; `StandardScaler` centraliza e escala pela dispersão; `MinMaxScaler` mapeia o intervalo observado; `IsolationForest` estima anomalias. **Nunca ajuste transformações usando o conjunto de teste**: ajuste no treino e só transforme o teste. Em produção, prefira `Pipeline`.

In [15]:
try:
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, MinMaxScaler
    from sklearn.ensemble import IsolationForest
except ImportError:
    print('Se desejar executar esta seção, instale scikit-learn no kernel.')
else:
    X_treino = np.array([[1.0], [2.0], [np.nan], [4.0]])
    X_teste = np.array([[np.nan], [5.0]])
    imputador = SimpleImputer(strategy='median')
    treino_imputado = imputador.fit_transform(X_treino)
    teste_imputado = imputador.transform(X_teste)
    scaler = StandardScaler().fit(treino_imputado)
    treino_escalado = scaler.transform(treino_imputado)
    teste_escalado = scaler.transform(teste_imputado)
    minmax = MinMaxScaler().fit(treino_imputado)
    print('Treino padronizado:', treino_escalado.ravel())
    print('Teste escalado:', teste_escalado.ravel())
    print('MinMax:', minmax.transform(teste_imputado).ravel())
    detector = IsolationForest(random_state=42, contamination=0.1)
    dados = np.array([[10.0], [11.0], [10.5], [11.5], [12.0], [100.0]])
    print('Detecção exploratória:', detector.fit_predict(dados))

Treino padronizado: [-1.14707867 -0.22941573 -0.22941573  1.60591014]
Teste escalado: [-0.22941573  2.52357307]
MinMax: [0.33333333 1.33333333]
Detecção exploratória: [ 1  1  1  1  1 -1]


### 9.7. Exercícios resolvidos

**Exercício 1 — Auditoria de dados:** crie uma função que devolva, para cada coluna, a quantidade de valores ausentes e o percentual correspondente.

In [16]:
def relatorio_ausentes(tabela):
    total = len(tabela)
    faltantes = tabela.isna().sum()
    porcentagem = (faltantes / total * 100) if total else faltantes * 0.0
    return pd.DataFrame({'ausentes': faltantes, 'percentual': porcentagem})

relatorio = relatorio_ausentes(df)
assert relatorio.loc['idade', 'ausentes'] == 2
print(relatorio)

        ausentes  percentual
pessoa         0    0.000000
idade          2   33.333333
renda          0    0.000000
grupo          0    0.000000
data           0    0.000000


**Exercício 2 — Atributos derivados:** crie uma coluna com a faixa de renda, outra com o mês e remova registros duplicados completos, sem alterar o DataFrame de origem.

In [17]:
def preparar_tabela(tabela):
    nova = tabela.drop_duplicates().copy()
    nova['faixa_renda'] = pd.cut(nova['renda'], bins=[0, 4000, 6000, float('inf')], labels=['baixa', 'média', 'alta'], include_lowest=True)
    nova['mes'] = pd.to_datetime(nova['data']).dt.month
    return nova

preparada = preparar_tabela(df)
assert 'faixa_renda' in preparada and 'mes' in preparada
assert len(preparada) == 5 and len(df) == 6
print(preparada[['pessoa', 'faixa_renda', 'mes']])

  pessoa faixa_renda  mes
0    Ana       baixa    1
1    Bia       média    2
2   Caio       média    3
4   Davi        alta    4
5    Eva       baixa    5


### 9.8. Consulta Rápida — tratamento de dados

| Técnica | Ferramenta | Cuidado importante |
|---|---|---|
| Nulos | `isna`, `fillna`, `dropna`, `SimpleImputer` | Imputar pode alterar a distribuição |
| Duplicações | `duplicated`, `drop_duplicates` | Definir o identificador de evento |
| Faixas | `pd.cut`, `pd.qcut` | Limites e categorias têm significado |
| Datas | `pd.to_datetime`, `.dt`, `tz_localize`, `tz_convert` | Distinguir horário local e instante UTC |
| Escala | `StandardScaler`, `MinMaxScaler` | Ajustar somente com treino |
| Anomalias | IQR, `IsolationForest` | Investigar antes de excluir |

## 10. Exercício integrador — rastreabilidade do ETL

**Enunciado:** compare os números de linhas de entrada, duplicatas removidas, saídas transformadas e registros com receita ausente. Confirme que as saídas derivadas preservam as contagens esperadas.

**Solução comentada:** o relatório explicita o que foi descartado por deduplicação e o que permanece incompleto. Em aplicações reais, registre também identificadores dos registros afetados e versões da fonte.

In [18]:
auditoria_etl = pd.Series({
    'entrada': len(brutos),
    'duplicatas_exatas': duplicatas_exatas,
    'apos_deduplicacao': len(tratados),
    'sem_preco': int(tratados['preco_unitario'].isna().sum()),
    'sem_referencia': int(integrado['_merge'].ne('both').sum()),
})
assert auditoria_etl['entrada'] == auditoria_etl['duplicatas_exatas'] + auditoria_etl['apos_deduplicacao']
assert resumo['registros'].sum() == auditoria_etl['apos_deduplicacao']
print(auditoria_etl.to_string())

entrada              49
duplicatas_exatas     1
apos_deduplicacao    48
sem_preco             1
sem_referencia        1


## Consulta Rápida — ETL reproduzível

| Etapa | Ferramenta | Pergunta de auditoria |
|---|---|---|
| Extração | `pd.read_csv(..., parse_dates=...)` | A fonte e os tipos foram identificados? |
| Esquema | Colunas obrigatórias, `pd.to_numeric`, `pd.to_datetime` | Há campos ausentes ou inválidos? |
| Qualidade | `isna`, `duplicated`, `between` | Quais registros exigem investigação? |
| Integração | `merge(validate='many_to_one', indicator=True)` | Houve multiplicação de linhas ou chaves sem correspondência? |
| Agregação | `groupby`, `sum(min_count=1)` | Total conhecido está sendo confundido com total real? |
| Saída | `to_csv`, `read_csv`, contagens e checks | É possível reproduzir e verificar o arquivo? |

**Erros comuns:** eliminar registros sem justificativa, substituir preço ausente por zero, executar fit no conjunto de teste ou salvar derivados em `datasets/`.

## 10. Aprofundamento v2 — contratos e integridade de ETL

O caminho `datasets/` representa **entrada imutável**. Escreva somente em `outputs/`, com validações antes e depois da exportação. Uma junção de muitos-para-um deve ter chaves únicas na tabela de referência.

**Exercício resolvido:** testar um contrato simples de colunas e conferir que a soma de unidades se mantém depois de serializar dados.

In [19]:
from pathlib import Path
import pandas as pd
entrada_v2 = DATA / 'vendas_exemplo.csv'
assert entrada_v2.is_file()
v2_bruto = pd.read_csv(entrada_v2)
campos_obrigatorios = {'data', 'regiao', 'produto', 'unidades', 'preco_unitario'}
assert campos_obrigatorios.issubset(v2_bruto.columns)
assert pd.to_numeric(v2_bruto['unidades'], errors='coerce').notna().all()
OUT = OUTPUTS / 'auditoria_v2.csv'
relatorio_v2 = pd.DataFrame({'campo':v2_bruto.columns, 'ausentes':v2_bruto.isna().sum().to_numpy()})
relatorio_v2.to_csv(OUT,index=False)
conferido_v2 = pd.read_csv(OUT)
assert relatorio_v2.equals(conferido_v2)
print('Contrato básico válido; relatório gerado somente em outputs/:', OUT.name)

Contrato básico válido; relatório gerado somente em outputs/: auditoria_v2.csv


**Limite:** os testes de colunas não substituem a definição de regras de negócio (faixas admissíveis, chaves compostas, duplicatas legítimas e convenções para dados ausentes).